In [1]:
#pip install cptac
import pandas as pd
import cptac

In [ ]:
protein = pd.read_csv("data/egfr_protein.csv")
rna = pd.read_csv("data/egfr_rna.csv")
full = pd.read_csv("data/egfr_full_merged.csv")
mutation = pd.read_csv("data/egfr_mutation.csv")
phospho = pd.read_csv("data/egfr_phospho_cleaned.csv")

protein = protein.rename(columns={"PATIENT_ID": "patient_id"})
rna = rna.rename(columns={"PATIENT_ID": "patient_id"})
full = full.rename(columns={"PATIENT_ID": "patient_id"})
mutation = mutation.rename(columns={"PATIENT_ID": "patient_id"})
phospho = phospho.rename(columns={"PATIENT_ID": "patient_id"})

#couldn't get patient ids to merger between tcga and cptac..alt is to identify data cohort

protein["cohort"] = "CPTAC"
protein["batch_domain"] = "CPTAC"

rna["cohort"] = "CPTAC"
rna["batch_domain"] = "CPTAC"

full["cohort"] = "CPTAC"
full["batch_domain"] = "CPTAC"

phospho["cohort"] = "CPTAC"
phospho["batch_domain"] = "CPTAC"

mutation["cohort"] = "TCGA"
mutation["batch_domain"] = "TCGA"

print(full.head())
print(mutation.head())

  patient_id  EGFR_PROTEIN  EGFR_RNA  EGFR_activity_mean cohort batch_domain
0  C3L-00001     28.192875     17.07                   1  CPTAC        CPTAC
1  C3L-00009     25.219585     11.86                   1  CPTAC        CPTAC
2  C3L-00080     25.238803     12.58                   1  CPTAC        CPTAC
3  C3L-00083     25.041583     10.94                   1  CPTAC        CPTAC
4  C3L-00093     24.469511     12.78                   1  CPTAC        CPTAC
        patient_id                mutation EGFR_type cohort batch_domain
0  TCGA-05-4382-01             R222L E545Q     Other   TCGA         TCGA
1  TCGA-05-4402-01  T751_I759delinsN I759N    Exon19   TCGA         TCGA
2  TCGA-05-4410-01                   R377S     Other   TCGA         TCGA
3  TCGA-05-5423-01             L833F L861Q     Other   TCGA         TCGA
4  TCGA-17-Z026-01                   G721V     Other   TCGA         TCGA


In [5]:
expression_protein_features = full[[
    "patient_id",
    "EGFR_PROTEIN",
    "EGFR_RNA",
    "EGFR_activity_mean",
    "cohort",
    "batch_domain"
]].copy()

print(expression_protein_features.head())

  patient_id  EGFR_PROTEIN  EGFR_RNA  EGFR_activity_mean cohort batch_domain
0  C3L-00001     28.192875     17.07                   1  CPTAC        CPTAC
1  C3L-00009     25.219585     11.86                   1  CPTAC        CPTAC
2  C3L-00080     25.238803     12.58                   1  CPTAC        CPTAC
3  C3L-00083     25.041583     10.94                   1  CPTAC        CPTAC
4  C3L-00093     24.469511     12.78                   1  CPTAC        CPTAC


In [7]:
mutation_features = mutation[[
    "patient_id",
    "mutation",
    "EGFR_type",
    "cohort",
    "batch_domain"
]].copy()

# Optional simple cleanup
mutation_features["gene"] = "EGFR"

print(mutation_features.head())

        patient_id                mutation EGFR_type cohort batch_domain  gene
0  TCGA-05-4382-01             R222L E545Q     Other   TCGA         TCGA  EGFR
1  TCGA-05-4402-01  T751_I759delinsN I759N    Exon19   TCGA         TCGA  EGFR
2  TCGA-05-4410-01                   R377S     Other   TCGA         TCGA  EGFR
3  TCGA-05-5423-01             L833F L861Q     Other   TCGA         TCGA  EGFR
4  TCGA-17-Z026-01                   G721V     Other   TCGA         TCGA  EGFR


In [8]:
phospho_value_cols = [c for c in phospho.columns if c not in ["patient_id", "cohort", "batch_domain"]]

phospho_reshaped = phospho.melt(
    id_vars=["patient_id", "cohort", "batch_domain"],
    value_vars=phospho_value_cols,
    var_name="EGFR_binding_site",
    value_name="EGFR_phospho_value"
)
phospho_reshaped = phospho_reshaped.dropna(subset=["EGFR_phospho_value"]).copy()

# Remove duplicate suffixes like .1
phospho_reshaped["EGFR_binding_site"] = phospho_reshaped["EGFR_binding_site"].str.replace(r"\.1$", "", regex=True)

# Keep only site label before underscore
phospho_reshaped["EGFR_binding_site"] = phospho_reshaped["EGFR_binding_site"].str.split("_").str[0]

# Collapse duplicate patient-site rows by averaging
phospho_dupl = (
    phospho_reshaped
    .groupby(["patient_id", "EGFR_binding_site", "cohort", "batch_domain"], as_index=False)["EGFR_phospho_value"]
    .mean()
)

print(phospho_reshaped.head())
print(phospho_dupl["EGFR_binding_site"].unique())

  patient_id cohort batch_domain EGFR_binding_site  EGFR_phospho_value
0  C3L-00001  CPTAC        CPTAC              S991           23.423220
1  C3L-00009  CPTAC        CPTAC              S991           22.750016
2  C3L-00080  CPTAC        CPTAC              S991           21.850419
3  C3L-00083  CPTAC        CPTAC              S991           21.916126
4  C3L-00093  CPTAC        CPTAC              S991           22.427414
['S1039' 'S1042' 'S1064' 'S1166' 'S991' 'T693' 'Y1172' 'Y1197' 'S1071'
 'Y1092' 'S1026' 'S695' 'T1041' 'S1045' 'S1025' 'T993' 'S1081' 'Y1069'
 'Y1016' 'T1085' 'Y1110']


In [10]:
all_patient_ids = pd.Series(
    pd.concat([
        expression_protein_features["patient_id"],
        mutation_features["patient_id"],
        phospho_reshaped["patient_id"]
    ]).unique(),
    name="patient_id"
)

master_samples = pd.DataFrame(all_patient_ids)

master_samples["has_mutation"] = master_samples["patient_id"].isin(mutation_features["patient_id"]).astype(int)
master_samples["has_rna"] = master_samples["patient_id"].isin(expression_protein_features["patient_id"]).astype(int)
master_samples["has_protein"] = master_samples["patient_id"].isin(expression_protein_features["patient_id"]).astype(int)
master_samples["has_phospho"] = master_samples["patient_id"].isin(phospho_reshaped["patient_id"]).astype(int)

master_samples["cohort"] = master_samples["patient_id"].apply(
    lambda x: "CPTAC" if x.startswith("C3L-") else "TCGA"
)
master_samples["batch_domain"] = master_samples["cohort"]

print(master_samples.head())
print(master_samples["cohort"].value_counts())

  patient_id  has_mutation  has_rna  has_protein  has_phospho cohort  \
0  C3L-00001             0        1            1            1  CPTAC   
1  C3L-00009             0        1            1            1  CPTAC   
2  C3L-00080             0        1            1            1  CPTAC   
3  C3L-00083             0        1            1            1  CPTAC   
4  C3L-00093             0        1            1            1  CPTAC   

  batch_domain  
0        CPTAC  
1        CPTAC  
2        CPTAC  
3        CPTAC  
4        CPTAC  
cohort
TCGA     210
CPTAC     67
Name: count, dtype: int64


In [20]:
master_samples.to_csv("data/master_samples.csv", index=False)
expression_protein_features.to_csv("data/expression_protein_features.csv", index=False)
mutation_features.to_csv("data/mutation_features.csv", index=False)
phospho_reshaped.to_csv("data/phospho_features.csv", index=False)

In [ ]:
#model ready..need hotspots
mutation = pd.read_csv("data/egfr_mutation.csv").rename(columns={"PATIENT_ID": "patient_id"})

mutation["cohort"] = "TCGA"
mutation["batch_domain"] = "TCGA"
mutation["gene"] = "EGFR"

# Keep original strings clean
mutation["mutation"] = mutation["mutation"].astype(str).str.strip()
mutation["EGFR_type"] = mutation["EGFR_type"].astype(str).str.strip()

print(mutation.head())
print(mutation.columns)

        patient_id                mutation EGFR_type cohort batch_domain  gene
0  TCGA-05-4382-01             R222L E545Q     Other   TCGA         TCGA  EGFR
1  TCGA-05-4402-01  T751_I759delinsN I759N    Exon19   TCGA         TCGA  EGFR
2  TCGA-05-4410-01                   R377S     Other   TCGA         TCGA  EGFR
3  TCGA-05-5423-01             L833F L861Q     Other   TCGA         TCGA  EGFR
4  TCGA-17-Z026-01                   G721V     Other   TCGA         TCGA  EGFR
Index(['patient_id', 'mutation', 'EGFR_type', 'cohort', 'batch_domain',
       'gene'],
      dtype='object')


In [22]:
#quick for loop (might need help for changing to not get docked points)

def label_egfr_hotspot(mutation_value: str) -> str:
    m = str(mutation_value).upper()

    if "EXON 19" in m or "19DEL" in m or "DEL19" in m:
        return "exon19del"
    elif "L858R" in m:
        return "L858R"
    elif "T790M" in m:
        return "T790M"
    elif "C797S" in m:
        return "C797S"
    elif "KINASE" in m:
        return "uncommon_kinase_domain"
    else:
        return "other"

mutation["egfr_hotspot_label"] = mutation["mutation"].apply(label_egfr_hotspot)
mutation["is_egfr_hotspot"] = mutation["egfr_hotspot_label"].ne("other").astype(int)

def classify_mutation(row) -> str:
    text = f"{row['mutation']} {row['EGFR_type']}".upper()

    if "MISSENSE" in text:
        return "missense"
    elif "NONSENSE" in text or "STOP" in text:
        return "nonsense"
    elif "FRAMESHIFT" in text:
        return "frameshift"
    elif "SPLICE" in text:
        return "splice"
    elif "AMP" in text or "AMPLIFICATION" in text:
        return "amplification"
    elif "DEL" in text or "DELETION" in text:
        return "deletion"
    elif "SYNONYMOUS" in text or "SILENT" in text:
        return "synonymous"
    else:
        return "other"

mutation["mutation_class"] = mutation.apply(classify_mutation, axis=1)

In [19]:
nonsynonymous_classes = {
    "missense",
    "nonsense",
    "frameshift",
    "splice",
    "amplification",
    "deletion"
}

mutation["is_nonsynonymous"] = mutation["mutation_class"].isin(nonsynonymous_classes).astype(int)


mutation_features = mutation[[
    "patient_id",
    "gene",
    "mutation",
    "EGFR_type",
    "mutation_class",
    "is_nonsynonymous",
    "is_egfr_hotspot",
    "egfr_hotspot_label",
    "cohort",
    "batch_domain"
]].copy()

print(mutation_features.head())
print(mutation_features["mutation_class"].value_counts(dropna=False))
print(mutation_features["egfr_hotspot_label"].value_counts(dropna=False))
mutation_features.to_csv("data/mutation_features_cleaned.csv", index=False)

        patient_id  gene                mutation EGFR_type mutation_class  \
0  TCGA-05-4382-01  EGFR             R222L E545Q     Other          other   
1  TCGA-05-4402-01  EGFR  T751_I759delinsN I759N    Exon19       deletion   
2  TCGA-05-4410-01  EGFR                   R377S     Other          other   
3  TCGA-05-5423-01  EGFR             L833F L861Q     Other          other   
4  TCGA-17-Z026-01  EGFR                   G721V     Other          other   

   is_nonsynonymous  is_egfr_hotspot egfr_hotspot_label cohort batch_domain  
0                 0                0              other   TCGA         TCGA  
1                 1                0              other   TCGA         TCGA  
2                 0                0              other   TCGA         TCGA  
3                 0                0              other   TCGA         TCGA  
4                 0                0              other   TCGA         TCGA  
mutation_class
other       43
deletion    26
splice       1
Name: cou

In [21]:
print(mutation_features[["mutation", "EGFR_type", "mutation_class", "egfr_hotspot_label"]].head(20))

                  mutation EGFR_type mutation_class egfr_hotspot_label
0              R222L E545Q     Other          other              other
1   T751_I759delinsN I759N    Exon19       deletion              other
2                    R377S     Other          other              other
3              L833F L861Q     Other          other              other
4                    G721V     Other          other              other
5             E746_A750del    Exon19       deletion              other
6                    L858R     L858R          other              L858R
7           D770_N771insGL     Other          other              other
8               L62R L858R     L858R          other              L858R
9             E746_A750del    Exon19       deletion              other
10            E746_A750del    Exon19       deletion              other
11            A767_V769dup     Other          other              other
12             G719C S768I     Other          other              other
13    